In [1]:
import pandas as pd
import numpy as np
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

BASE_DIR = ".."
DATA_PATH = os.path.join(BASE_DIR, "data")
SUB_PATH = os.path.join(BASE_DIR, "submissions")

def reduce_mem_usage(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

print("Setup Optimized!")

Setup Optimized!


In [2]:
# Load data dengan optimasi memori
df_train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df_test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

# Langsung kecilkan ukuran memori
df_train = reduce_mem_usage(df_train)
df_test = reduce_mem_usage(df_test)

print(f"Data Loaded! Memory reduced.")

Data Loaded! Memory reduced.


In [3]:
def build_advanced_features(df):
    # 1. Handling Timestamp
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['hour'] = df['Timestamp'].dt.hour
    df['day_of_week'] = df['Timestamp'].dt.dayofweek
    
    # 2. Sensor Magnitude (Vector Sum) - Deteksi Safety_Violation
    df['accel_mag'] = np.sqrt(df['Accel_X']**2 + df['Accel_Y']**2 + df['Accel_Z']**2)
    
    # 3. Price & Distance Logic (Anomaly Detection)
    # Fitur Rasio Harga per KM
    df['price_per_km'] = df['Est_Price_IDR'] / (df['Distance_KM'] + 0.1)
    
    # Fitur Anomali Jarak (Khusus data 0-1 KM yang tadi kita diskusikan)
    df['is_short_trip'] = (df['Distance_KM'] <= 1.0).astype(int)
    df['high_price_short_trip'] = ((df['Distance_KM'] <= 1.0) & (df['Est_Price_IDR'] > 50000)).astype(int)
    
    # 4. Device Brand Extraction
    # Mengambil kata pertama saja (misal: Samsung-Hot -> Samsung)
    df['Device_Brand'] = df['Device_FP'].astype(str).apply(lambda x: x.split('-')[0] if '-' in x else x)
    
    # 5. Missing Data Signal
    # Menghitung berapa banyak kolom yang null dalam satu baris
    df['null_count'] = df.isnull().sum(axis=1)
    
    # Drop kolom mentah yang sudah diekstrak
    cols_to_drop = ['Timestamp', 'Trip_ID'] if 'Trip_ID' in df.columns else ['Timestamp']
    return df.drop(columns=cols_to_drop)

y = df_train['Trip_Label']
X = build_advanced_features(df_train.drop(columns=['Trip_Label']))
X_test_final = build_advanced_features(df_test)

print("Advanced Features Created!")

Advanced Features Created!


In [4]:
# Ambil 25% data (2 Juta baris) - Aman untuk RAM 16GB
X_samp, _, y_samp, _ = train_test_split(
    X, y, train_size=0.25, stratify=y, random_state=42
)

# Identifikasi Categorical Features
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
X_samp[cat_features] = X_samp[cat_features].fillna("MISSING").astype(str)
X_test_final[cat_features] = X_test_final[cat_features].fillna("MISSING").astype(str)

# Split Train & Validasi
X_train, X_val, y_train, y_val = train_test_split(
    X_samp, y_samp, test_size=0.15, stratify=y_samp, random_state=42
)

# Definisi Bobot Manual (Tweak agar Macro F1 naik tanpa merusak Precision)
# Memberi perhatian lebih pada label yang tadinya 0.00
class_weights = {
    'Perfect_Trip': 1.0,
    'Safety_Violation': 1.5,
    'Fraud_Indication': 2.0,
    'Navigation_Issue': 3.5,  # Dinaikkan karena tersulit
    'Service_Complaint': 3.5  # Dinaikkan karena tersulit
}

print("Sampling & Manual Weights Ready!")

Sampling & Manual Weights Ready!


In [6]:
model = CatBoostClassifier(
    iterations=1500, # Tambah iterasi tapi ada early stopping
    learning_rate=0.04,
    depth=7, # Depth 7 lebih cepat dan generalis dibanding 8
    task_type="CPU",
    thread_count=12,
    loss_function='MultiClass',
    eval_metric='TotalF1', # Fokus pada F1 selama training
    class_weights=class_weights, # Menggunakan bobot manual kita
    early_stopping_rounds=100,
    random_seed=42,
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 0.4294127	test: 0.4291409	best: 0.4291409 (0)	total: 8.99s	remaining: 3h 44m 42s
100:	learn: 0.4370521	test: 0.4368477	best: 0.4368818 (98)	total: 15m 50s	remaining: 3h 39m 25s
200:	learn: 0.4371718	test: 0.4369576	best: 0.4370479 (163)	total: 36m 17s	remaining: 3h 54m 31s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.4370479482
bestIteration = 163

Shrink model to first 164 iterations.


In [7]:
y_val_pred = model.predict(X_val).flatten()
macro_f1 = f1_score(y_val, y_val_pred, average="macro")

print(f"\nTARGET ACHIEVED? MACRO F1: {macro_f1:.4f}")
print("-" * 30)
print(classification_report(y_val, y_val_pred, zero_division=0))


TARGET ACHIEVED? MACRO F1: 0.5558
------------------------------
                   precision    recall  f1-score   support

 Fraud_Indication       0.96      0.98      0.97     15012
 Navigation_Issue       0.00      0.00      0.00     30067
     Perfect_Trip       0.72      1.00      0.84    164910
 Safety_Violation       0.99      0.95      0.97     60060
Service_Complaint       0.00      0.00      0.00     29951

         accuracy                           0.79    300000
        macro avg       0.54      0.59      0.56    300000
     weighted avg       0.64      0.79      0.70    300000



In [8]:
test_preds = model.predict(X_test_final).flatten()
submission = pd.DataFrame({
    "Trip_ID": df_test["Trip_ID"],
    "Trip_Label": test_preds
})

sub_file = os.path.join(SUB_PATH, "submission_manual_weights_v2.csv")
submission.to_csv(sub_file, index=False)
print(f"File siap di-submit: {sub_file}")

File siap di-submit: ../submissions/submission_manual_weights_v2.csv
